# 1. Imports

In [63]:
%load_ext autoreload
%autoreload 2

import os
import requests
import glob
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 500)
pd.options.plotting.backend = "plotly"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 2. Data : Télécharger les fichiers sources depuis PRIM IDFM

In [ ]:
# Télécharger les fichiers sources depuis la Plateforme Régionale d'Information pour la Mobilité (PRIM) d'Ile-de-France Mobilités et enregistrer localement.

# Données de validation de titres par trimestre.

url_validations_t1 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-1er-trimestre/exports/csv" 

url_validations_t2 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-2eme-trimestre/exports/csv" 

url_validations_t3 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-3eme-trimestre/exports/csv"

url_validations_t4 = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/validations-reseau-ferre-nombre-validations-par-jour-4eme-trimestre/exports/csv"


# Chemin du répertoire pour mettre les fichiers csv de validation.
folder_path_validations_multiples = os.path.join("..","data","raw","validations_multiples")

# Vérifier et créer le répertoire de destination s'il n'existe pas.

if not os.path.exists(folder_path_validations_multiples):
    os.makedirs(folder_path_validations_multiples)
    print(f"Répertoire créé : {folder_path_validations_multiples}")



# Liste et emplacements des gares / stations.

url_stations = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/emplacement-des-gares-idf-data-generalisee/exports/csv" 


# Chemin du répertoire pour mettre les fichiers csv de liste et emplacement des gares / stations.
folder_path_stations = os.path.join("..","data","raw","stations")

# Vérifier et créer le répertoire de destination s'il n'existe pas.

if not os.path.exists(folder_path_stations):
    os.makedirs(folder_path_stations)
    print(f"Répertoire créé : {folder_path_stations}")



# Définir une fonction pour télécharger les sources.

def telecharger_csv(url, destination, fichier):
    # Chemin complet du fichier.
    filepath = os.path.join(destination, fichier)
    try:
        print(f"Téléchargement en cours depuis : {url}")

        # Effectuer une requête HTTP.
        reponse = requests.get(url, timeout=10)

        # Erreur si le statut HTTP n'est pas bon (404, 500).
        reponse.raise_for_status()

        # Ecrire le contenu dans un fichier (en mode binaire).
        with open(filepath, "wb") as f:
            f.write(reponse.content)

        print(f"Fichier enregistré sous : {filepath}")

    except requests.exceptions.RequestException as e:
        print(f"Erreur lors du téléchargement : {e}")


if __name__ == "__main__":

    # Télécharger les fichiers de validations.
    
    url = url_validations_t1
    destination = folder_path_validations_multiples
    fichier = "validations_t1.csv"

    telecharger_csv(url, destination, fichier)


    url = url_validations_t2
    destination = folder_path_validations_multiples
    fichier = "validations_t2.csv"

    telecharger_csv(url, destination, fichier)

    url = url_validations_t3
    destination = folder_path_validations_multiples
    fichier = "validations_t3.csv"

    telecharger_csv(url, destination, fichier)


    url = url_validations_t4
    destination = folder_path_validations_multiples
    fichier = "validations_t4.csv"

    telecharger_csv(url, destination, fichier)


    # Télécharger les fichiers de liste et emplacements des gares / stations.

    url = url_stations
    destination = folder_path_stations
    fichier = "stations.csv"

    telecharger_csv(url, destination, fichier)



# 3. EDA

## 3.1. Données de validation de titres.

In [64]:
# Objectif : combiner les fichiers sources de validations en un seul fichier csv.

# Nom de colonne "ida" pour "validations_t1.csv" alors que les autres csv sont en "id_zdc"
filepath_t1 = os.path.join("..","data","raw","validations_multiples","validations_t1.csv")
df_t1 = pd.read_csv(filepath_t1,sep=";")
df_t1

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald
0,2025-03-07,100,110,487,MADELEINE,71324.0,Forfaits courts,3387
1,2025-03-07,100,110,49,BARBES-ROCH.,71426.0,Autres titres,478
2,2025-03-07,100,110,490763,S. GAINSBOURG,490779.0,Contrat Solidarité Transport,781
3,2025-03-07,100,110,490763,S. GAINSBOURG,490779.0,Forfait Navigo,2880
4,2025-03-07,100,110,490763,S. GAINSBOURG,490779.0,Imagine R,1158
...,...,...,...,...,...,...,...,...
469371,2025-03-01,800,853,485,LUZARCHES,67177.0,Forfait Navigo,47
469372,2025-03-01,800,853,536,MERIEL,67065.0,Amethyste,2
469373,2025-03-01,800,853,536,MERIEL,67065.0,Imagine R,50
469374,2025-03-01,800,853,568,MONTSOULT,67000.0,Forfait Navigo,286


In [65]:
# Correction nom de colonne "ida" en "id_zdc"
df_t1 = df_t1.rename(columns={"ida": "id_zdc"})
df_t1

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald
0,2025-03-07,100,110,487,MADELEINE,71324.0,Forfaits courts,3387
1,2025-03-07,100,110,49,BARBES-ROCH.,71426.0,Autres titres,478
2,2025-03-07,100,110,490763,S. GAINSBOURG,490779.0,Contrat Solidarité Transport,781
3,2025-03-07,100,110,490763,S. GAINSBOURG,490779.0,Forfait Navigo,2880
4,2025-03-07,100,110,490763,S. GAINSBOURG,490779.0,Imagine R,1158
...,...,...,...,...,...,...,...,...
469371,2025-03-01,800,853,485,LUZARCHES,67177.0,Forfait Navigo,47
469372,2025-03-01,800,853,536,MERIEL,67065.0,Amethyste,2
469373,2025-03-01,800,853,536,MERIEL,67065.0,Imagine R,50
469374,2025-03-01,800,853,568,MONTSOULT,67000.0,Forfait Navigo,286


In [66]:
# Enregistrer en csv
filepath_validations_multiples = os.path.join("..","data","raw","validations_multiples","validations_t1.csv")
df_t1.to_csv(filepath_validations_multiples,sep=';',index=False, encoding='utf-8-sig')

In [67]:

# 4 fichiers csv de validations de titres (1 par trimestre) pour 2025 à combiner

# Chemin du répertoire où se trouvent les fichiers csv à combiner
folder_path_validations_multiples = os.path.join("..","data","raw","validations_multiples")

# Récuperer tous les fichiers csv dans ce répertoire
validations = glob.glob(os.path.join(folder_path_validations_multiples, "*.csv"))

# List comprehension pour lire et combiner / concaténer
validations_df = pd.concat((pd.read_csv(f, sep=";") for f in validations), ignore_index=True)


In [68]:
# Chemin du répertoire pour mettre le fichiers csv de validation combiné.
folder_path_validations = os.path.join("..","data","raw","validations")

# Vérifier et créer le répertoire de destination s'il n'existe pas.
if not os.path.exists(folder_path_validations):
    os.makedirs(folder_path_validations)
    print(f"Répertoire créé : {folder_path_validations}")

# Exporter vers un fichier csv.
combined_validations_filepath = os.path.join("..","data","raw","validations","validations.csv")
validations_df.to_csv(combined_validations_filepath, index=False)

print(f"Les {len(validations)} fichiers csv sont combinés avec succès dans : {combined_validations_filepath} !")


Les 4 fichiers csv sont combinés avec succès dans : ../data/raw/validations/validations.csv !


### 3.1.1. Vue globale des données.

In [69]:
validations_df.head()

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald
0,2025-03-07,100.0,110,487,MADELEINE,71324.0,Forfaits courts,3387.0
1,2025-03-07,100.0,110,49,BARBES-ROCH.,71426.0,Autres titres,478.0
2,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779.0,Contrat Solidarité Transport,781.0
3,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779.0,Forfait Navigo,2880.0
4,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779.0,Imagine R,1158.0


Colonnes interessantes pour l'analyse : jour, id_zdc pour faire les jointures, categorie_titre, nb_vald.

In [70]:
print(validations_df.shape)
print(f"Données de validation : Nombre de ligne : {validations_df.shape[0]}, Nombre de colonnes : {validations_df.shape[1]} ")

(1892453, 8)
Données de validation : Nombre de ligne : 1892453, Nombre de colonnes : 8 


In [71]:
validations_df.columns

Index(['jour', 'code_stif_trns', 'code_stif_res', 'code_stif_arret',
       'libelle_arret', 'id_zdc', 'categorie_titre', 'nb_vald'],
      dtype='object')

In [72]:
validations_df.describe()

,code_stif_trns,id_zdc,nb_vald
count,1.892453e+06,1.890831e+06,1.892453e+06
mean,5.020467e+02,9.026463e+04,1.140139e+03
std,3.472100e+02,8.951037e+04,3.084966e+03
min,1.000000e+02,-1.000000e+00,1.000000e+00
25%,1.000000e+02,6.653500e+04,4.400000e+01
50%,8.000000e+02,7.115000e+04,2.150000e+02
75%,8.000000e+02,7.192000e+04,1.016000e+03
max,8.100000e+02,9.999990e+05,9.997000e+04


In [73]:
validations_df.describe(include=object)

,jour,code_stif_res,code_stif_arret,libelle_arret,categorie_titre
count,1892453,1892453,1892453,1892453,1892453
unique,365,18,787,779,8
top,2025-11-04,110,48093,GARE DE LYON,Forfait Navigo
freq,5365,806414,8905,7665,284117


In [74]:
validations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892453 entries, 0 to 1892452
Data columns (total 8 columns):
 #   Column           Dtype  
---  ------           -----  
 0   jour             object 
 1   code_stif_trns   float64
 2   code_stif_res    object 
 3   code_stif_arret  object 
 4   libelle_arret    object 
 5   id_zdc           float64
 6   categorie_titre  object 
 7   nb_vald          float64
dtypes: float64(3), object(5)
memory usage: 115.5+ MB


In [75]:
validations_df["categorie_titre"].value_counts()

categorie_titre
Forfait Navigo                  284117
Imagine R                       282766
Forfaits courts                 280786
Amethyste                       270264
NON DEFINI                      248620
Autres titres                   244447
Contrat Solidarité Transport    210239
Contrat Solidarite Transport     71214
Name: count, dtype: int64

Doublon catégorie pour "Solidarité" et "Solidarite"  
NON DEFINI désigne une anomalie chez idfm. Ne pas grouper avec "Autres titres".

### 3.1.2. Data cleaning.

In [76]:
# Formater colonne "jour".
validations_df["jour"] = pd.to_datetime(validations_df["jour"])

In [77]:
validations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892453 entries, 0 to 1892452
Data columns (total 8 columns):
 #   Column           Dtype         
---  ------           -----         
 0   jour             datetime64[ns]
 1   code_stif_trns   float64       
 2   code_stif_res    object        
 3   code_stif_arret  object        
 4   libelle_arret    object        
 5   id_zdc           float64       
 6   categorie_titre  object        
 7   nb_vald          float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 115.5+ MB


In [78]:
# Ajout colonnes année.
#validations_df["annee"] = validations_df["jour"].dt.year

# Ajout colonnes mois.
validations_df["mois"] = validations_df["jour"].dt.month

# Ajout colonnes jour.
#validations_df["jour"] = validations_df["jour"].dt.day

# Ajout colonnes jour de la semaine (0 pour Lundi et 6 pour Dimanche).
validations_df["jour_sem_num"] = validations_df["jour"].dt.weekday

# Ajout colonnes nom du jour.
#validations_df["jour_nom"] = validations_df["jour"].dt.day_name()

# Ajout colonne nom du mois.
#validations_df["mois_nom"] = validations_df["jour"].dt.month_name()

In [79]:
validations_df.head()

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald,mois,jour_sem_num
0,2025-03-07,100.0,110,487,MADELEINE,71324.0,Forfaits courts,3387.0,3,4
1,2025-03-07,100.0,110,49,BARBES-ROCH.,71426.0,Autres titres,478.0,3,4
2,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779.0,Contrat Solidarité Transport,781.0,3,4
3,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779.0,Forfait Navigo,2880.0,3,4
4,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779.0,Imagine R,1158.0,3,4


In [80]:
# Correction "categorie_titre": renommer "Contrat Solidarité Transport" par "Contrat Solidarite Transport"
validations_df["categorie_titre"] = validations_df["categorie_titre"].replace("Contrat Solidarité Transport", "Contrat Solidarite Transport")

In [81]:
validations_df["categorie_titre"].value_counts()

categorie_titre
Forfait Navigo                  284117
Imagine R                       282766
Contrat Solidarite Transport    281453
Forfaits courts                 280786
Amethyste                       270264
NON DEFINI                      248620
Autres titres                   244447
Name: count, dtype: int64

In [82]:
# Valeurs manquantes.
validations_df.isna().sum()

jour                  0
code_stif_trns        0
code_stif_res         0
code_stif_arret       0
libelle_arret         0
id_zdc             1622
categorie_titre       0
nb_vald               0
mois                  0
jour_sem_num          0
dtype: int64

1622 valeurs manquantes sur 2 millions pour la colonne id_zdc

In [83]:
# A quoi correspondent les valeurs manquantes ?
validations_df[validations_df["id_zdc"].isnull()]

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald,mois,jour_sem_num
469921,2025-09-22,100.0,ND,ND,Inconnu,NaN,Amethyste,155.0,9,0
472867,2025-09-19,100.0,ND,ND,Inconnu,NaN,Amethyste,177.0,9,4
472868,2025-09-19,100.0,ND,ND,Inconnu,NaN,Contrat Solidarite Transport,683.0,9,4
472869,2025-09-19,100.0,ND,ND,Inconnu,NaN,Imagine R,1388.0,9,4
474528,2025-09-24,100.0,ND,ND,Inconnu,NaN,Forfaits courts,537.0,9,2
...,...,...,...,...,...,...,...,...,...,...
1408195,2025-06-27,100.0,ND,ND,Inconnu,NaN,Forfaits courts,5.0,6,4
1408196,2025-06-27,760.0,760,00490909,Aéroport d'Orly,NaN,Contrat Solidarite Transport,841.0,6,4
1409197,2025-06-30,760.0,760,00490909,Aéroport d'Orly,NaN,Imagine R,1493.0,6,0
1409198,2025-06-30,760.0,760,00490909,Aéroport d'Orly,NaN,NON DEFINI,366.0,6,0


In [84]:
# Attribuer une "id_zdc" à "Aéroport d'Orly" (information depuis la table stations) 
validations_df.loc[validations_df["libelle_arret"] == "Aéroport d'Orly", "id_zdc"] = 63284

In [85]:
# Supprimer les lignes dont "libelle_arret" == "Inconnu" (séléctionner les lignes où le libellé est différent de "Inconnu" et écraser l'acien dataframe).
validations_df = validations_df[validations_df["libelle_arret"] != "Inconnu"]

In [86]:
# Modifier id_zdc de float64 en int64.
validations_df["id_zdc"] = validations_df["id_zdc"].astype(int)

In [87]:
# Modifier id_zdc de int64 en str.
validations_df["id_zdc"] = validations_df["id_zdc"].astype(str)

In [88]:
# Correction "id_zdc" de validations pour correspondre aux "id_ref_zdc" des stations.

dictionnaire_id_zdc = {
    '59577' : '478855',
    '62737' : '478505',
    '63650' : '463850',
    '67747' : '462934',
    '69531' : '463754',
    '71219' : '473829',
    '71245' : '71229',
    '71282' : '479068',
    '71686' : '71697',
    '71743' : '463564',
    '72059' : '478883',
    '72219' : '72225',
    '73616' : '478885',
    '73792' : '478926',
    '73794' : '474151',
    '73795' : '474152',
    '74040' : '71139',
    '74371' : '463843',
    '93066' : '490784',
    '412697' : '479919',
    '425819' : '66915',
    '474149' : '73615',
    '474150' : '71229',
    '482368' : '73688',
    '492980' : '71271'    
}

validations_df["id_zdc"] = validations_df["id_zdc"].replace(dictionnaire_id_zdc)

In [89]:
# Modifier id_zdc Magenta par 478733
validations_df.loc[(validations_df["id_zdc"] == '74000') & (validations_df["libelle_arret"] == "MAGENTA"), "id_zdc"] = '478733'

In [90]:
validations_df[validations_df["libelle_arret"] == 'MAGENTA']

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald,mois,jour_sem_num
454,2025-03-07,800.0,805,490,MAGENTA,478733,NON DEFINI,782.0,3,4
1511,2025-03-08,800.0,805,490,MAGENTA,478733,Amethyste,157.0,3,5
1512,2025-03-08,800.0,805,490,MAGENTA,478733,Autres titres,183.0,3,5
2730,2025-01-01,800.0,805,490,MAGENTA,478733,Forfait Navigo,1507.0,1,2
3751,2025-01-02,800.0,805,490,MAGENTA,478733,Autres titres,444.0,1,3
...,...,...,...,...,...,...,...,...,...,...
1887276,2025-10-06,800.0,805,490,MAGENTA,478733,NON DEFINI,880.0,10,0
1887626,2025-10-07,800.0,805,490,MAGENTA,478733,Forfaits courts,2980.0,10,1
1889266,2025-10-03,800.0,805,490,MAGENTA,478733,Forfaits courts,3569.0,10,4
1890050,2025-10-04,800.0,805,490,MAGENTA,478733,Autres titres,71.0,10,5


In [91]:
# Modifier id_zdc La Chapelle par 71434
validations_df.loc[(validations_df["id_zdc"] == '74000') & (validations_df["libelle_arret"] == "LA CHAPELLE"), "id_zdc"] = '71434'

In [92]:
validations_df[validations_df["libelle_arret"] == 'LA CHAPELLE']

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald,mois,jour_sem_num
1017,2025-03-08,100.0,110,409,LA CHAPELLE,71434,NON DEFINI,54.0,3,5
2230,2025-01-01,100.0,110,409,LA CHAPELLE,71434,NON DEFINI,44.0,1,2
3261,2025-01-02,100.0,110,409,LA CHAPELLE,71434,Imagine R,719.0,1,3
4410,2025-01-05,100.0,110,409,LA CHAPELLE,71434,Amethyste,396.0,1,6
5423,2025-01-06,100.0,110,409,LA CHAPELLE,71434,Amethyste,583.0,1,0
...,...,...,...,...,...,...,...,...,...,...
1890694,2025-10-09,100.0,110,409,LA CHAPELLE,71434,Forfaits courts,1522.0,10,3
1890695,2025-10-09,100.0,110,409,LA CHAPELLE,71434,Imagine R,1770.0,10,3
1891824,2025-10-07,100.0,110,409,LA CHAPELLE,71434,Autres titres,29.0,10,1
1891825,2025-10-07,100.0,110,409,LA CHAPELLE,71434,Forfait Navigo,5638.0,10,1


In [93]:
# Pas de doublons.
validations_df.duplicated().sum()

np.int64(0)

## 3.2. Liste et emplacement des gares / stations.

In [94]:
# Liste des stations généralisée (une station peut regrouper plusieurs lignes de transport associées)
filepath_stations = os.path.join("..","data","raw","stations","stations.csv")
stations_df = pd.read_csv(filepath_stations, sep=";")

### 3.2.1. Vue globale des données.

In [95]:
stations_df.head()

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y
0,"48.88218913760796, 2.70601418111842","{""coordinates"": [2.70601418111842, 48.88218913...",5,3,Lagny-Thorigny,NaN,NaN,68494,Lagny - Thorigny,427872,Lagny - Thorigny,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,678439.8038,6.864726e+06
1,"48.83964551726303, 2.655130086641314","{""coordinates"": [2.655130086641314, 48.8396455...",7,7,Torcy,NaN,NaN,68129,Torcy,43207,Torcy,A01856,C01742,RER A,RER,0,1,0,0,0,0,0,0,0,0,RATP,1,0,674687.4504,6.860010e+06
2,"48.79565482051668, 2.6503876698084805","{""coordinates"": [2.650387669808481, 48.7956548...",9,10,Roissy-en-Brie,NaN,NaN,67839,Roissy-en-Brie,46568,Roissy-en-Brie,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,674317.7148,6.855121e+06
3,"48.77077891616031, 2.690252782201096","{""coordinates"": [2.690252782201096, 48.7707789...",10,11,Ozoir-la-Ferrière,NaN,NaN,462934,Ozoir-la-Ferrière,462901,Ozoir-la-Ferrière,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,677235.3132,6.852342e+06
4,"48.960226490735955, 2.9500603827528087","{""coordinates"": [2.950060382752809, 48.9602264...",13,16,Trilport,NaN,NaN,68984,Trilport,47962,Trilport,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,696343.0315,6.873365e+06


Colonnes interessantes pour l'analyse :  
geo_point_2d pour carte  
id_ref_zdc pour les jointures  
nom_zdc  
res_com  
mode  
train, rer, metro, tramway, val pour identifier les modes  
tertrain, terrer, termetro, tertram, terval pour voir l'impact des terminus  
exploitant  
principal 

In [96]:
print(stations_df.shape)
print(f"Données de validation : Nombre de ligne : {stations_df.shape[0]}, Nombre de colonnes : {stations_df.shape[1]} ")

(999, 30)
Données de validation : Nombre de ligne : 999, Nombre de colonnes : 30 


In [97]:
stations_df.columns

Index(['geo_point_2d', 'geo_shape', 'objectid_1', 'codeunique', 'nom_long',
       'nom_so_gar', 'nom_su_gar', 'id_ref_zdc', 'nom_zdc', 'id_ref_zda',
       'nom_zda', 'idrefliga', 'idrefligc', 'res_com', 'mode', 'train', 'rer',
       'metro', 'tramway', 'val', 'tertrain', 'terrer', 'termetro', 'tertram',
       'terval', 'exploitant', 'idf', 'principal', 'x', 'y'],
      dtype='object')

In [98]:
stations_df.describe().T

,count,mean,std,min,25%,50%,75%,max
objectid_1,999.0,5.361041e+02,336.213435,-1.000000,2.265000e+02,5.470000e+02,8.345000e+02,1.101000e+03
codeunique,999.0,3.987828e+04,52563.103408,1.000000,2.585000e+02,5.210000e+02,1.070280e+05,1.171150e+05
id_ref_zdc,999.0,1.108205e+05,117408.947771,59403.000000,6.733850e+04,7.122300e+04,7.246600e+04,4.909190e+05
id_ref_zda,999.0,1.214172e+05,160126.428183,42142.000000,4.328400e+04,4.563000e+04,5.401650e+04,4.940200e+05
train,999.0,2.692693e-01,0.443802,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
rer,999.0,2.402402e-01,0.427443,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
metro,999.0,3.213213e-01,0.467218,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
tramway,999.0,2.742743e-01,0.446371,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
val,999.0,1.001001e-02,0.099598,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
tertrain,999.0,3.503504e-02,0.183960,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00


In [99]:
stations_df.describe(include=object).T

,count,unique,top,freq
geo_point_2d,999,999,"48.88218913760796, 2.70601418111842",1
geo_shape,999,999,"{""coordinates"": [2.70601418111842, 48.88218913...",1
nom_long,999,997,Saint-Fargeau,2
nom_so_gar,68,65,Hôtel de Ville,2
nom_su_gar,29,12,VITRY-SUR-SEINE,6
nom_zdc,999,990,NC,3
nom_zda,999,994,Saint-Fargeau,2
idrefliga,999,188,A01842,49
idrefligc,999,190,C01728,49
res_com,999,186,RER D,49


In [100]:
stations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 30 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   geo_point_2d  999 non-null    object 
 1   geo_shape     999 non-null    object 
 2   objectid_1    999 non-null    int64  
 3   codeunique    999 non-null    int64  
 4   nom_long      999 non-null    object 
 5   nom_so_gar    68 non-null     object 
 6   nom_su_gar    29 non-null     object 
 7   id_ref_zdc    999 non-null    int64  
 8   nom_zdc       999 non-null    object 
 9   id_ref_zda    999 non-null    int64  
 10  nom_zda       999 non-null    object 
 11  idrefliga     999 non-null    object 
 12  idrefligc     999 non-null    object 
 13  res_com       999 non-null    object 
 14  mode          999 non-null    object 
 15  train         999 non-null    int64  
 16  rer           999 non-null    int64  
 17  metro         999 non-null    int64  
 18  tramway       999 non-null    

### 3.2.2. Data cleaning.

In [101]:
# Création Colonnes latitude et longitude.
stations_df[["latitude","longitude"]] = stations_df["geo_point_2d"].str.split(", ", expand=True).astype(float)

In [102]:
stations_df.head()

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y,latitude,longitude
0,"48.88218913760796, 2.70601418111842","{""coordinates"": [2.70601418111842, 48.88218913...",5,3,Lagny-Thorigny,NaN,NaN,68494,Lagny - Thorigny,427872,Lagny - Thorigny,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,678439.8038,6.864726e+06,48.882189,2.706014
1,"48.83964551726303, 2.655130086641314","{""coordinates"": [2.655130086641314, 48.8396455...",7,7,Torcy,NaN,NaN,68129,Torcy,43207,Torcy,A01856,C01742,RER A,RER,0,1,0,0,0,0,0,0,0,0,RATP,1,0,674687.4504,6.860010e+06,48.839646,2.655130
2,"48.79565482051668, 2.6503876698084805","{""coordinates"": [2.650387669808481, 48.7956548...",9,10,Roissy-en-Brie,NaN,NaN,67839,Roissy-en-Brie,46568,Roissy-en-Brie,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,674317.7148,6.855121e+06,48.795655,2.650388
3,"48.77077891616031, 2.690252782201096","{""coordinates"": [2.690252782201096, 48.7707789...",10,11,Ozoir-la-Ferrière,NaN,NaN,462934,Ozoir-la-Ferrière,462901,Ozoir-la-Ferrière,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,677235.3132,6.852342e+06,48.770779,2.690253
4,"48.960226490735955, 2.9500603827528087","{""coordinates"": [2.950060382752809, 48.9602264...",13,16,Trilport,NaN,NaN,68984,Trilport,47962,Trilport,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,696343.0315,6.873365e+06,48.960226,2.950060


In [103]:
# Modifier id_zdc en str.
stations_df["id_ref_zdc"] = stations_df["id_ref_zdc"].astype(str)

In [104]:
# Valeurs manquantes.
stations_df.isna().sum()

geo_point_2d      0
geo_shape         0
objectid_1        0
codeunique        0
nom_long          0
nom_so_gar      931
nom_su_gar      970
id_ref_zdc        0
nom_zdc           0
id_ref_zda        0
nom_zda           0
idrefliga         0
idrefligc         0
res_com           0
mode              0
train             0
rer               0
metro             0
tramway           0
val               0
tertrain          0
terrer            0
termetro          0
tertram           0
terval            0
exploitant        0
idf               0
principal         0
x                 0
y                 0
latitude          0
longitude         0
dtype: int64

Pas de valeurs manquantes sauf pour nom_so_gar (nom sous gare) et nom_su_gar (nom sur gare) qui est normal : toutes les stations n'ont pas forcémment de sur/sous nom.

In [105]:
# Remplacer dans "termetro": "METRO 14" par 1 (valeur 1 ou 0 uniquement).
stations_df.loc[stations_df["termetro"] == "METRO 14", "termetro"] = 1

In [106]:
# Modifier termetro de object à int64. 
stations_df["termetro"] = stations_df["termetro"].astype(int)

In [109]:
# Correction de noms de stations et res_com.

stations_df.loc[stations_df["id_ref_zdc"] == '478733', "nom_zdc"] = "Magenta"
stations_df.loc[stations_df["id_ref_zdc"] == '478855', "nom_zdc"] = "Etampes"
stations_df.loc[stations_df["id_ref_zdc"] == '71697', "nom_zdc"] = "Avron"
stations_df.loc[stations_df["id_ref_zdc"] == '479928', "nom_zdc"] = "Buzenval"
stations_df.loc[stations_df["id_ref_zdc"] == '73688', "nom_zdc"] = "Havre-Caumartin"



stations_df.loc[stations_df["id_ref_zdc"] == '71697', "res_com"] = "METRO 9"
stations_df.loc[stations_df["id_ref_zdc"] == '479928', "res_com"] = "METRO 2"
stations_df.loc[stations_df["id_ref_zdc"] == '73688', "res_com"] = "METRO 3 / METRO 9"


# Correction Châtelet les Halles.

stations_df.loc[(stations_df["id_ref_zdc"] == '474151') & (stations_df["nom_zdc"] == "Châtelet les Halles"), "res_com"] = "RER A / RER B / RER D / METRO 4"
stations_df.loc[(stations_df["id_ref_zdc"] == '474151') & (stations_df["nom_zdc"] == "Châtelet les Halles"), "mode"] = "RER / METRO"
stations_df.loc[(stations_df["id_ref_zdc"] == '474151') & (stations_df["nom_zdc"] == "Châtelet les Halles"), "metro"] = 1
stations_df["metro"] = stations_df["metro"].astype(int)


In [110]:
stations_df[stations_df["id_ref_zdc"] == '474151']

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y,latitude,longitude
477,"48.86251523497548, 2.345785860039181","{""coordinates"": [2.345785860039181, 48.8625152...",355,104051,Les Halles,NaN,NaN,474151,Les Halles,45102,Châtelet les Halles,A01537,C01374,METRO 4,METRO,0,0,1,0,0,0,0,0,0,0,RATP,1,0,652003.8587,6.862697e+06,48.862515,2.345786
846,"48.86182227190629, 2.347012688612925","{""coordinates"": [2.347012688612925, 48.8618222...",298,326,Châtelet-Les Halles,NaN,NaN,474151,Châtelet les Halles,45102,Châtelet les Halles,A01856 / A01857 / A01842,C01742 / C01743 / C01728,RER A / RER B / RER D / METRO 4,RER / METRO,0,1,1,0,0,0,0,0,0,0,RATP,1,1,652093.2252,6.862619e+06,48.861822,2.347013


In [111]:
# Modifier id_ref_zdc Château Landon par 73615
stations_df.loc[(stations_df["id_ref_zdc"] == '71359') & (stations_df["nom_zdc"] == "Château Landon"), "id_ref_zdc"] = '73615'

In [112]:
stations_df[(stations_df["id_ref_zdc"] == '73615') & (stations_df["nom_zdc"] == "Château Landon")]

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y,latitude,longitude
174,"48.87844430039445, 2.3619653022299603","{""coordinates"": [2.36196530222996, 48.87844430...",982,107043,Château-Landon,NaN,NaN,73615,Château Landon,479053,Château Landon,A01540,C01377,METRO 7,METRO,0,0,1,0,0,0,0,0,0,0,RATP,1,0,653205.1424,6.864458e+06,48.878444,2.361965


In [113]:
# Suppression des doublons id_ref_zdc

stations_df = stations_df[~((stations_df["id_ref_zdc"] == '63284') & (stations_df["nom_zdc"] == "Aéroport Orly 1-2-3 (Terminal Ouest)"))]
stations_df = stations_df[~((stations_df["id_ref_zdc"] == '69677') & (stations_df["nom_zdc"] == "Pont de Rungis Aéroport d'Orly"))]
stations_df = stations_df[~((stations_df["id_ref_zdc"] == '71229') & (stations_df["nom_zdc"] == "La Muette"))]
stations_df = stations_df[~((stations_df["id_ref_zdc"] == '71607') & (stations_df["nom_zdc"] == "Gare de Bercy"))]
stations_df = stations_df[~((stations_df["id_ref_zdc"] == '73620') & (stations_df["nom_zdc"] == "Cluny La Sorbonne"))]
stations_df = stations_df[~((stations_df["id_ref_zdc"] == '73753') & (stations_df["nom_zdc"] == "Porte de Thiais (Marché international)"))]
stations_df = stations_df[~((stations_df["id_ref_zdc"] == '474151') & (stations_df["nom_zdc"] == "Les Halles"))]

In [114]:
# Ajout colonne "nb_lignes" nombre de ligne selon contenu de "res_com".
stations_df["nb_lignes"] = stations_df["res_com"].str.count("/") + 1

In [115]:
stations_df.head()

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y,latitude,longitude,nb_lignes
0,"48.88218913760796, 2.70601418111842","{""coordinates"": [2.70601418111842, 48.88218913...",5,3,Lagny-Thorigny,NaN,NaN,68494,Lagny - Thorigny,427872,Lagny - Thorigny,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,678439.8038,6.864726e+06,48.882189,2.706014,1
1,"48.83964551726303, 2.655130086641314","{""coordinates"": [2.655130086641314, 48.8396455...",7,7,Torcy,NaN,NaN,68129,Torcy,43207,Torcy,A01856,C01742,RER A,RER,0,1,0,0,0,0,0,0,0,0,RATP,1,0,674687.4504,6.860010e+06,48.839646,2.655130,1
2,"48.79565482051668, 2.6503876698084805","{""coordinates"": [2.650387669808481, 48.7956548...",9,10,Roissy-en-Brie,NaN,NaN,67839,Roissy-en-Brie,46568,Roissy-en-Brie,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,674317.7148,6.855121e+06,48.795655,2.650388,1
3,"48.77077891616031, 2.690252782201096","{""coordinates"": [2.690252782201096, 48.7707789...",10,11,Ozoir-la-Ferrière,NaN,NaN,462934,Ozoir-la-Ferrière,462901,Ozoir-la-Ferrière,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,677235.3132,6.852342e+06,48.770779,2.690253,1
4,"48.960226490735955, 2.9500603827528087","{""coordinates"": [2.950060382752809, 48.9602264...",13,16,Trilport,NaN,NaN,68984,Trilport,47962,Trilport,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,696343.0315,6.873365e+06,48.960226,2.950060,1


## 3.3. Merge

In [116]:
# Merge
df = validations_df.merge(stations_df, how='left', left_on='id_zdc', right_on='id_ref_zdc')

In [117]:
df.head()

,jour,code_stif_trns,code_stif_res,code_stif_arret,libelle_arret,id_zdc,categorie_titre,nb_vald,mois,jour_sem_num,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y,latitude,longitude,nb_lignes
0,2025-03-07,100.0,110,487,MADELEINE,71324,Forfaits courts,3387.0,3,4,"48.87023324339993, 2.3251635468278975","{""coordinates"": [2.325163546827897, 48.8702332...",650.0,108039.0,Madeleine,NaN,NaN,71324,Madeleine,43898.0,Madeleine,A01541 / A01545 / A01547,C01378 / C01382 / C01384,METRO 8 / METRO 12 / METRO 14,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,650498.2822,6.863568e+06,48.870233,2.325164,3.0
1,2025-03-07,100.0,110,49,BARBES-ROCH.,71426,Autres titres,478.0,3,4,"48.88373439813717, 2.349525203221073","{""coordinates"": [2.349525203221073, 48.8837343...",1016.0,104037.0,Barbès-Rochechouart,NaN,NaN,71426,Barbès Rochechouart,42280.0,Barbès Rochechouart,A01535 / A01537,C01372 / C01374,METRO 2 / METRO 4,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,652297.6246,6.865054e+06,48.883734,2.349525,2.0
2,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779,Contrat Solidarite Transport,781.0,3,4,"48.881316062234305, 2.4271855777339173","{""coordinates"": [2.427185577733917, 48.8813160...",0.0,117104.0,Serge Gainsbourg,NaN,NaN,490779,Serge Gainsbourg,490763.0,Serge Gainsbourg,A01544,C01381,METRO 11,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,657990.7676,6.864741e+06,48.881316,2.427186,1.0
3,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779,Forfait Navigo,2880.0,3,4,"48.881316062234305, 2.4271855777339173","{""coordinates"": [2.427185577733917, 48.8813160...",0.0,117104.0,Serge Gainsbourg,NaN,NaN,490779,Serge Gainsbourg,490763.0,Serge Gainsbourg,A01544,C01381,METRO 11,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,657990.7676,6.864741e+06,48.881316,2.427186,1.0
4,2025-03-07,100.0,110,490763,S. GAINSBOURG,490779,Imagine R,1158.0,3,4,"48.881316062234305, 2.4271855777339173","{""coordinates"": [2.427185577733917, 48.8813160...",0.0,117104.0,Serge Gainsbourg,NaN,NaN,490779,Serge Gainsbourg,490763.0,Serge Gainsbourg,A01544,C01381,METRO 11,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,657990.7676,6.864741e+06,48.881316,2.427186,1.0


In [ ]:
# Supprimer les colonnes non utiles.
colonnes_a_supprimer = ["code_stif_trns", "code_stif_res", "code_stif_arret", "nom_long", "nom_so_gar", "nom_su_gar", "geo_point_2d", "geo_shape", "objectid_1", "codeunique", "id_ref_zda", "nom_zda", "idrefliga", "idrefligc", "x", "y"]

df = df.drop(columns=colonnes_a_supprimer)


In [119]:
df.head()

,jour,libelle_arret,id_zdc,categorie_titre,nb_vald,mois,jour_sem_num,id_ref_zdc,nom_zdc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,latitude,longitude,nb_lignes
0,2025-03-07,MADELEINE,71324,Forfaits courts,3387.0,3,4,71324,Madeleine,METRO 8 / METRO 12 / METRO 14,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,48.870233,2.325164,3.0
1,2025-03-07,BARBES-ROCH.,71426,Autres titres,478.0,3,4,71426,Barbès Rochechouart,METRO 2 / METRO 4,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,48.883734,2.349525,2.0
2,2025-03-07,S. GAINSBOURG,490779,Contrat Solidarite Transport,781.0,3,4,490779,Serge Gainsbourg,METRO 11,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,48.881316,2.427186,1.0
3,2025-03-07,S. GAINSBOURG,490779,Forfait Navigo,2880.0,3,4,490779,Serge Gainsbourg,METRO 11,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,48.881316,2.427186,1.0
4,2025-03-07,S. GAINSBOURG,490779,Imagine R,1158.0,3,4,490779,Serge Gainsbourg,METRO 11,METRO,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RATP,1.0,0.0,48.881316,2.427186,1.0


In [120]:
df.shape

(1890468, 27)

In [121]:
df.columns

Index(['jour', 'libelle_arret', 'id_zdc', 'categorie_titre', 'nb_vald', 'mois',
       'jour_sem_num', 'id_ref_zdc', 'nom_zdc', 'res_com', 'mode', 'train',
       'rer', 'metro', 'tramway', 'val', 'tertrain', 'terrer', 'termetro',
       'tertram', 'terval', 'exploitant', 'idf', 'principal', 'latitude',
       'longitude', 'nb_lignes'],
      dtype='object')

In [122]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1890468 entries, 0 to 1890467
Data columns (total 27 columns):
 #   Column           Dtype         
---  ------           -----         
 0   jour             datetime64[ns]
 1   libelle_arret    object        
 2   id_zdc           object        
 3   categorie_titre  object        
 4   nb_vald          float64       
 5   mois             int32         
 6   jour_sem_num     int32         
 7   id_ref_zdc       object        
 8   nom_zdc          object        
 9   res_com          object        
 10  mode             object        
 11  train            float64       
 12  rer              float64       
 13  metro            float64       
 14  tramway          float64       
 15  val              float64       
 16  tertrain         float64       
 17  terrer           float64       
 18  termetro         float64       
 19  tertram          float64       
 20  terval           float64       
 21  exploitant       object        

In [123]:
# Modifier "nb_vald" de float64 en int64.
df["nb_vald"] = df["nb_vald"].astype(int)

In [124]:
# List comprehension pour calculer le nombre de validations par lignes.

lignes = ['RER A', 'RER B', 'RER C', 'RER D', 'RER E', 'METRO 1', 'METRO 2', 'METRO 3', 'METRO 3bis','METRO 4', 'METRO 5', 'METRO 6', 'METRO 7', 'METRO 7bis','METRO 8', 'METRO 9', 'METRO 10', 'METRO 11', 'METRO 12', 'METRO 13', 'METRO 14', 'TRAIN H', 'TRAIN J', 'TRAIN K', 'TRAIN L', 'TRAIN N', 'TRAIN P', 'TRAIN R', 'TRAIN V', 'TRAM 1', 'TRAM 2', 'TRAM 3', 'TRAM 3a', 'TRAM 3b', 'TRAM 4', 'TRAM 5', 'TRAM 6', 'TRAM 7', 'TRAM 8', 'TRAM 9', 'TRAM 10', 'TRAM 11', 'TRAM 12', 'TRAM 13', 'TRAM 14', 'CABLE 1', 'CDGVAL', 'FUNICULAIRE MONTMARTRE']

lignes_df = pd.DataFrame({
    "Ligne": lignes,
    "somme_nb_vald": [df[df["res_com"].str.contains(l, na=False)]["nb_vald"].sum() for l in lignes]
})

lignes_df

,Ligne,somme_nb_vald
0,RER A,324994656
1,RER B,190203361
2,RER C,180974086
3,RER D,191192832
4,RER E,201652557
5,METRO 1,842868704
6,METRO 2,127083079
7,METRO 3,187142264
8,METRO 3bis,11019527
9,METRO 4,301538185


In [ ]:
# Validations par ligne et par catégorie de titre (NE PAS UTILISER CAR ERREUR RESULTAT).

#lignes = ['RER A', 'RER B', 'RER C', 'RER D', 'RER E', 'METRO 1', 'METRO 2', 'METRO 3', 'METRO 3bis','METRO 4', 'METRO 5', 'METRO 6', 'METRO 7', 'METRO 7bis','METRO 8', 'METRO 9', 'METRO 10', 'METRO 11', 'METRO 12', 'METRO 13', 'METRO 14', 'TRAIN H', 'TRAIN J', 'TRAIN K', 'TRAIN L', 'TRAIN N', 'TRAIN P', 'TRAIN R', 'TRAIN V', 'TRAM 1', 'TRAM 2', 'TRAM 3', 'TRAM 3a', 'TRAM 3b', 'TRAM 4', 'TRAM 5', 'TRAM 6', 'TRAM 7', 'TRAM 8', 'TRAM 9', 'TRAM 10', 'TRAM 11', 'TRAM 12', 'TRAM 13', 'TRAM 14', 'CABLE 1', 'CDGVAL', 'FUNICULAIRE MONTMARTRE']

#lignes_df = (
#   df[df["res_com"].isin(lignes)]
#   .groupby(["res_com", "categorie_titre"])["nb_vald"]
#   .sum()
#   .reset_index()
#)

#lignes_df

# 4. Export

Exporter les fichiers en format csv.

In [125]:
# Sauvegarder les données de validations traitées dans un nouveau csv.

processed_validations_filepath = os.path.join('..','data','processed','validations.csv')
validations_df.to_csv(processed_validations_filepath, index=False, encoding='utf-8-sig')

print(f"Fichier 'données de validations' enregistré sous : {processed_validations_filepath}")

Fichier 'données de validations' enregistré sous : ../data/processed/validations.csv


In [126]:
# Sauvegarder les données de stations traitées dans un nouveau csv.

processed_stations_filepath = os.path.join('..','data','processed','stations.csv')
stations_df.to_csv(processed_stations_filepath, index=False, encoding='utf-8-sig')

print(f"Fichier 'liste des stations' enregistré sous : {processed_stations_filepath}")

Fichier 'liste des stations' enregistré sous : ../data/processed/stations.csv


In [127]:
# Sauvegarder les données fusionnées dans un nouveau csv.

processed_fusion_filepath = os.path.join('..','data','processed','validations_fusion.csv')
df.to_csv(processed_fusion_filepath, index=False, encoding='utf-8-sig')

print(f"Fichier 'fusion valiations et stations' enregistré sous : {processed_fusion_filepath}")

Fichier 'fusion valiations et stations' enregistré sous : ../data/processed/validations_fusion.csv


In [128]:
# Sauvegarder les données de validations par ligne de transport dans un nouveau csv.

processed_ligne_filepath = os.path.join('..','data','processed','validations_ligne.csv')
lignes_df.to_csv(processed_ligne_filepath, index=False, encoding='utf-8-sig')

print(f"Fichier 'validations par ligne' enregistré sous : {processed_ligne_filepath}")

Fichier 'validations par ligne' enregistré sous : ../data/processed/validations_ligne.csv
